In [6]:
# @title Install & Import Dependencies
# 1. Install the missing libraries first
!pip install scenedetect[opencv] ultralytics --quiet

# 2. Now import them
import cv2
import os
from scenedetect import detect, ContentDetector, split_video_ffmpeg
from ultralytics import YOLO
from IPython.display import Video, display

print("Libraries installed and imported successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.9/130.9 kB 8.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
preprocessing 0.1.13 requires nltk==3.2.4, but you have nltk 3.9.2 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
Libraries installed and imported successfully.


In [7]:
# @title 1. Install SAM 2 and Dependencies
import os

# Install SAM 2 from the repository (same as your reference nb)
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git
!pip install -q supervision ultralytics

# Download the SAM 2 Checkpoint (Model Weights)
if not os.path.exists("sam2_hiera_large.pt"):
    print("Downloading SAM 2 Model...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

print("Installation Complete.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.4/212.4 kB 5.3 MB/s eta 0:00:00a 0:00:01
Installation Complete.


TASK - 2.1

In [2]:
!pip install scenedetect[opencv]
import os
import glob
import subprocess
from scenedetect import detect, split_video_ffmpeg
from scenedetect.detectors import AdaptiveDetector

# --- CONFIGURATION ---
input_video_path = "/kaggle/input/input-video/video.mp4" 
output_dir = "/kaggle/working/plays"

START_TIME = "00:40:00"    # Start extracting at Minute 40
DURATION_LIMIT = "00:03:00" # Extract for 3 Minutes (Ends at 43:00)

# --- 1. SAFETY CHECKS ---
if not os.path.exists(input_video_path):
    # Auto-find logic for Kaggle
    mp4s = glob.glob("/kaggle/input/**/*.mp4", recursive=True)
    if mp4s:
        input_video_path = mp4s[0]
        print(f"Found video: {input_video_path}")
    else:
        raise FileNotFoundError("Check your Input path!")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# --- 2. THE SPEED HACK: Pre-Trim the Video ---
# We create a temporary, smaller file to analyze
short_video_path = "/kaggle/working/short_game.mp4"

print(f"SPEED HACK: Jumping to {START_TIME} and extracting {DURATION_LIMIT}...")

# Uses ffmpeg to fast-forward to 40:00 and copy 3 mins
# -ss START_TIME (Seek to 40m)
# -t DURATION_LIMIT (Take 3 mins)
# -c copy (Instant processing, no re-encoding)
subprocess.run([
    "ffmpeg", "-y", "-i", input_video_path, 
    "-ss", START_TIME, 
    "-t", DURATION_LIMIT, 
    "-c", "copy", short_video_path
], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

print(f"Created {short_video_path} (From {START_TIME} to 43:00)")

# --- 3. EXECUTE CHUNKING ON THE SHORT CLIP ---
print(f"Detecting plays in the targeted clip...")

# Using AdaptiveDetector as requested (good for fast-paced sports)
# min_scene_len=90 is approx 3 seconds at 30fps
scene_list = detect(short_video_path, AdaptiveDetector(adaptive_threshold=3.0, min_scene_len=90))

print(f"Found {len(scene_list)} plays in this segment.")

if len(scene_list) > 0:
    print(f"Splitting {len(scene_list)} clips...")
    split_video_ffmpeg(short_video_path, scene_list, output_dir=output_dir, show_progress=False)
    
    print("\nDONE! Here are your plays:")
    plays = sorted(glob.glob(os.path.join(output_dir, "*.mp4")))
    for p in plays:
        print(f" - {os.path.basename(p)}")
else:
    print("No plays found in this segment. Try lowering the threshold or min_scene_len.")

SPEED HACK: Jumping to 00:40:00 and extracting 00:03:00...
Created /kaggle/working/short_game.mp4 (From 00:40:00 to 43:00)
Detecting plays in the targeted clip...
Found 10 plays in this segment.
Splitting 10 clips...

DONE! Here are your plays:
 - short_game-Scene-001.mp4
 - short_game-Scene-002.mp4
 - short_game-Scene-003.mp4
 - short_game-Scene-004.mp4
 - short_game-Scene-005.mp4
 - short_game-Scene-006.mp4
 - short_game-Scene-007.mp4
 - short_game-Scene-008.mp4
 - short_game-Scene-009.mp4
 - short_game-Scene-010.mp4


In [4]:
import os
import glob
import random
from IPython.display import Video, display

# --- CONFIGURATION ---
plays_folder = "/kaggle/working/plays"

# 1. Find all the clips
all_plays = glob.glob(os.path.join(plays_folder, "*.mp4"))

if not all_plays:
    print("No plays found! Did you run the chunking step above?")
else:
    # 2. Pick a random one
    random_play = random.choice(all_plays)
    play_name = os.path.basename(random_play)
    
    print(f"Randomly Selected: {play_name}")
    
    # 3. FIX FOR KAGGLE/BROWSER PLAYBACK
    # Browsers often fail to play videos unless they are strictly H.264 + yuv420p.
    # We create a temporary "viewable" copy to ensure it works.
    viewable_path = "/kaggle/working/viewable_play.mp4"
    
    # Fast re-encode to ensure compatibility
    # -y (overwrite) -vcodec libx264 (video format) -pix_fmt yuv420p (browser color format)
    os.system(f"ffmpeg -y -loglevel error -i '{random_play}' -vcodec libx264 -pix_fmt yuv420p '{viewable_path}'")
    
    # 4. Display
    display(Video(viewable_path, embed=True, width=600))

Randomly Selected: short_game-Scene-008.mp4


In [7]:
import shutil
import os
from IPython.display import FileLink

# 1. Zip the 'plays' folder
output_filename = "my_plays_backup"
dir_name = "/kaggle/working/plays"

print(" Zipping folder...")
shutil.make_archive(output_filename, 'zip', dir_name)

# 2. Create a clickable download link
print(f"Created {output_filename}.zip")
FileLink(r'my_plays_backup.zip')

 Zipping folder...
Created my_plays_backup.zip


/kaggle/working/my_plays_backup.zip

Task 2.2 carried forward in another ipynb due to numpy issue